In [4]:
import os
import pandas as pd 
from main_class import cls_frame
from pathlib import Path

In [5]:
current_folder = Path("/home/b0urb0n/code/container/evaluation/current")

outputs_folder = Path("/home/b0urb0n/code/container/evaluation/outputs")
validation_folder = Path("/home/b0urb0n/code/container/evaluation/validation")


records = []

In [3]:
cls_frame.eval(curretn_path=str(current_folder), outputs_folder=outputs_folder)



100%|██████████| 275/275 [00:07<00:00, 37.85it/s]


In [6]:

for subfolder in sorted(outputs_folder.iterdir()):
    name = subfolder.name
    val_folder = validation_folder / str(name)

    for class_ in sorted(subfolder.iterdir()):


        pred_class = class_.name

        for img_path in class_.glob("*.png"):
            filename = img_path.name

            val_matches = list(val_folder.rglob(filename))

            if val_matches:
                true_class = val_matches[0].parent.name  # Parent folder is the true class

                records.append({
                    "id": name,
                    "filename": filename,
                    "true_class": true_class,
                    "pred_class": pred_class,
                    "is_correct": true_class == pred_class
                })

df = pd.DataFrame(records)

In [7]:
df

,id,filename,true_class,pred_class,is_correct
0,2b32324d34ec46ff7eef8c6d1d54a0af7dbe19c62d3593...,frame_0016.png,b-mode,b-mode,True
1,2b32324d34ec46ff7eef8c6d1d54a0af7dbe19c62d3593...,frame_0027.png,b-mode,b-mode,True
2,2b32324d34ec46ff7eef8c6d1d54a0af7dbe19c62d3593...,frame_0043.png,b-mode,b-mode,True
3,2b32324d34ec46ff7eef8c6d1d54a0af7dbe19c62d3593...,frame_0019.png,b-mode,b-mode,True
4,2b32324d34ec46ff7eef8c6d1d54a0af7dbe19c62d3593...,frame_0072.png,b-mode,b-mode,True
...,...,...,...,...,...
761,anonymized_slm_casa-rpi_1_samsung_ws80a-71-202...,frame_0227.png,non-usable,non-usable,True
762,anonymized_slm_casa-rpi_1_samsung_ws80a-71-202...,frame_0171.png,non-usable,non-usable,True
763,anonymized_slm_casa-rpi_1_samsung_ws80a-71-202...,frame_0225.png,doppler-mode,non-usable,False
764,anonymized_slm_casa-rpi_1_samsung_ws80a-71-202...,frame_0172.png,non-usable,non-usable,True


In [8]:
df.to_csv("/home/b0urb0n/code/container/evaluation/summary.csv", index=False)

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score

# Ensure 'is_correct' exists
if "is_correct" not in df.columns:
    df["is_correct"] = df["true_class"] == df["pred_class"]

if df.empty:
    print("Error: DataFrame is empty. Check your folder paths and matching filenames.")
else:

    print("=== 1. PER-MACHINE ACCURACY PER CLASS ===")
    
    pivot_correct = df.groupby(["id", "true_class"])["is_correct"].sum()
    pivot_total = df.groupby(["id", "true_class"])["is_correct"].count()
    pivot_pct = (pivot_correct / pivot_total * 100).round(2)

    formatted_matrix = pd.DataFrame(index=pivot_total.index)
    formatted_matrix["display"] = (
        pivot_pct.astype(str) + "% (" + pivot_correct.astype(str) + "/" + pivot_total.astype(str) + ")"
    )
    
    machine_class_df = formatted_matrix["display"].unstack(level="true_class").fillna("N/A")
    print(machine_class_df.to_string())
    
    print("\n" + "="*60 + "\n")


    print("=== 2. AGGREGATE ACCURACY PER CLASS ===")
    agg_class_acc = df.groupby("true_class")["is_correct"].mean() * 100

    for class_name, acc in agg_class_acc.items():
        total_class_samples = (df["true_class"] == class_name).sum()
        correct_class_samples = ((df["true_class"] == class_name) & df["is_correct"]).sum()
        print(f"Class '{class_name}': {acc:.2f}% ({correct_class_samples}/{total_class_samples})")

    print("\n" + "="*60 + "\n")


    total_accuracy = accuracy_score(df["true_class"], df["pred_class"]) * 100
    correct_total = df["is_correct"].sum()
    total_samples = len(df)

    print("=== 3. ENTIRE MODEL OVERALL ACCURACY ===")
    print(f"Overall Accuracy: {total_accuracy:.2f}% ({correct_total}/{total_samples} total frames)")

=== 1. PER-MACHINE ACCURACY PER CLASS ===
true_class                                                                             b-mode    doppler-mode measurement-mode      non-usable    pw-doppler     split_frame
id                                                                                                                                                                          
2b32324d34ec46ff7eef8c6d1d54a0af7dbe19c62d35933d53c4d83d85e053f2.mkv_frames    100.0% (45/45)  80.77% (21/26)       0.0% (0/2)    100.0% (5/5)  100.0% (4/4)    100.0% (9/9)
2e3635e5cb0ee0c51d2a035e6dd23be179cb671aa9ca990d717abdc9af544f70.mkv_frames  100.0% (142/142)      0.0% (0/2)              N/A  100.0% (25/25)    0.0% (0/1)  37.04% (20/54)
anonymized_har-rpi_2_mindray_pediatrie-03-20231024075137.mkv_frames          100.0% (169/169)    100.0% (2/2)              N/A    33.33% (1/3)           N/A     50.0% (1/2)
anonymized_slm_casa-rpi_1_samsung_ws80a-71-20231017132211.mkv_frames         100.0% (268/268)